# Batch processing

This tutorial covers the `.many()` method for efficient bulk feature extraction:

- Plain Python lists of `(t, m, sigma)` tuples
- [nested-pandas](https://nested-pandas.readthedocs.io) with real ZTF survey data
- [PyArrow](https://arrow.apache.org/docs/python/) `List<Struct>` arrays
- [Polars](https://docs.pola.rs) Series

All Arrow-compatible inputs avoid Python-level iteration and pass data to Rust with zero copies.

In [ ]:
# %pip install "light-curve"

## Plain list of tuples

`.many()` accepts a list of `(t, m, sigma)` tuples and returns a 2-D NumPy array of shape
`(N, n_features)`. Multi-threading is enabled by default via the `n_jobs` parameter:

In [ ]:
import light_curve as licu
import numpy as np

rng = np.random.default_rng(0)
light_curves = [
    (np.sort(rng.random(50)), rng.random(50), rng.random(50) * 0.1)
    for _ in range(1000)
]

results = licu.Amplitude().many(light_curves)
print(f'Extracted from {len(light_curves)} light curves: shape = {results.shape}')
print(f'Mean amplitude = {results.mean():.4f} mag')

## nested-pandas with ZTF survey data

[nested-pandas](https://nested-pandas.readthedocs.io) extends pandas with nested Arrow column
support, useful for catalog data such as ZTF or Rubin LSST.

```sh
pip install light-curve nested-pandas s3fs universal-pathlib
```

In [ ]:
# %pip install "light-curve" nested-pandas s3fs universal-pathlib

In [ ]:
import light_curve as licu
import nested_pandas as npd
import numpy as np
import pyarrow as pa
from upath import UPath

s3_path = UPath(
    "s3://ipac-irsa-ztf/contributed/dr23/lc/hats/ztf_dr23_lc-hats/dataset/Norder=6/Dir=30000/Npix=34623/part0.snappy.parquet",
    anon=True,
)
ndf = npd.read_parquet(
    s3_path,
    columns=["objectid", "lightcurve.hmjd", "lightcurve.mag", "lightcurve.magerr"],
)

ndf = ndf.loc[ndf["lightcurve"].list_lengths > 10]

ndf["lightcurve.t"] = np.asarray(ndf["lightcurve.hmjd"] - 58000, dtype=np.float32)

feature = licu.Extractor(licu.Chi2Pvar(), licu.InterPercentileRange(quantile=0.25), licu.LinearFit())
result = feature.many(pa.array(ndf["lightcurve"]), n_jobs=-1,
                      arrow_fields={"t": "t", "m": "mag", "sigma": "magerr"})

ndf = ndf.assign(**dict(zip(feature.names, result.T)))
ndf.head()

## PyArrow

[PyArrow](https://arrow.apache.org/docs/python/) is the reference Python implementation of Apache Arrow.
Pass a `List<Struct<t, m, band>>` array directly to `.many()` for multiband extraction without sigma:

```sh
pip install light-curve pyarrow
```


In [ ]:
# %pip install "light-curve" pyarrow

In [ ]:
import light_curve as licu
import numpy as np
import pyarrow as pa

BANDS = ["g", "r"]
rng = np.random.default_rng(42)
n_lc, n_per_band = 200, 40

struct_type = pa.struct([
    ("t", pa.float64()),
    ("m", pa.float64()),
    ("band", pa.utf8()),
])


def make_lc():
    rows = []
    for b in BANDS:
        t = rng.uniform(0, 100, n_per_band)
        m = rng.normal(15.0 if b == "g" else 15.3, 0.3, n_per_band)
        rows.extend({"t": float(ti), "m": float(mi), "band": b} for ti, mi in zip(t, m))
    rows.sort(key=lambda r: r["t"])
    return rows


lcs_arrow = pa.array([make_lc() for _ in range(n_lc)], type=pa.list_(struct_type))

feature = licu.Extractor(
    licu.InterPercentileRange(quantile=0.1, bands=BANDS),  # robust amplitude per band
    licu.AndersonDarlingNormal(bands=BANDS),  # normality test per band
    licu.ColorOfMaximum(BANDS),  # colour at brightness peak
    licu.ColorOfMinimum(BANDS),  # colour at brightness trough
)
result = feature.many(
    lcs_arrow,
    sorted=True,
    arrow_fields={"t": "t", "m": "m", "band": "band"},
)
print(f"shape: {result.shape}")  # (200, 6)
print("names:", feature.names)


## Polars

[Polars](https://docs.pola.rs) is a fast DataFrame library built on Arrow.
Group a flat multiband DataFrame by object and pass the nested Series to `.many()`:

```sh
pip install light-curve polars
```


In [ ]:
# %pip install "light-curve" polars

In [ ]:
import light_curve as licu
import numpy as np
import polars as pl

BANDS = ["g", "r"]
rng = np.random.default_rng(42)
n_obj, n_per_band = 200, 40

object_id = np.repeat(np.arange(n_obj), n_per_band * len(BANDS))
band_col = np.tile(np.repeat(BANDS, n_per_band), n_obj)
t = np.sort(rng.uniform(0, 100, n_obj * n_per_band * len(BANDS)))
m = rng.normal(15.0, 0.3, len(object_id))
sigma = rng.uniform(0.01, 0.1, len(object_id))

df = pl.DataFrame({"object_id": object_id, "band": band_col, "t": t, "m": m, "sigma": sigma})
nested = df.group_by("object_id").agg(pl.struct("t", "m", "sigma", "band").alias("lc"))

feature = licu.Extractor(
    licu.ExcessVariance(bands=BANDS),  # variability excess over noise per band
    licu.StetsonK(bands=BANDS),  # variability index per band
    licu.BeyondNStd(nstd=1.5, bands=BANDS),  # outlier fraction per band
    licu.ColorOfMedian(BANDS),  # colour at median brightness
    licu.ColorSpread(BANDS),  # std dev of per-band means
)
result = feature.many(
    nested["lc"],
    arrow_fields={"t": "t", "m": "m", "sigma": "sigma", "band": "band"},
)
nested = nested.with_columns(
    [pl.Series(name, result[:, i]) for i, name in enumerate(feature.names)]
)
nested.select(["object_id"] + feature.names)


## Next steps

- [Feature basics tutorial](basics.ipynb) — single features, Extractor, multiband intro
- [Multiband tutorial](../multiband/) — per-band and cross-band features
- [Periodogram tutorial](../periodogram/) — Lomb–Scargle and period search
- [API reference](../api/) — full signatures and equations